In [1]:
from abiasales.data_generation import *
from abiasales.cleaning import *
from abiasales.weights import *
from abiasales.stats import *
from abiasales.stratification import *
from abiasales.metrics import *
from abiasales.inference import *
from abiasales.power import *
from abiasales.results import *
from abiasales.simulation import *

In [2]:
import warnings
warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:,.2f}'.format

# DATA

In [3]:
df_a = create_test_data(uplift=0, n_users=1000000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.1, n_users=1000000)
df_b['exp_group'] = 'B'

df = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column         Dtype  
---  ------         -----  
 0   uid            int64  
 1   platform       object 
 2   country        object 
 3   strata         object 
 4   clicks         float64
 5   payer_flag     int64  
 6   purchases      int64  
 7   views          float64
 8   views_cov      float64
 9   clicks_cov     float64
 10  purchases_cov  float64
 11  exp_group      object 
dtypes: float64(5), int64(3), object(4)
memory usage: 183.1+ MB


In [5]:
STRATS_COLS = ['platform', 'country']
FEATURE_COLS = ['views_cov', 'clicks_cov', 'purchases_cov']

# CLEANING

Handling outliers

Initial sample size

In [6]:
len(df)

2000000

Mean metrics which we will use for handling outliers

In [7]:
df.groupby('exp_group')[FEATURE_COLS].mean()

,views_cov,clicks_cov,purchases_cov
exp_group,,,
A,13.03,1.72,2.08
B,13.08,1.89,2.28


Removing outliers

In [8]:
df = remove_outliers(df, metrics=FEATURE_COLS, contamination=0.001)

Means became a little bit less

In [9]:
df.groupby('exp_group')[FEATURE_COLS].mean()

,views_cov,clicks_cov,purchases_cov
exp_group,,,
A,13.01,1.71,2.08
B,13.04,1.89,2.27


Checking that exactly 0.1% of samples were removed

In [10]:
len(df)

1998000

Capping values of metric

In [11]:
metrics_to_cap = ['purchases']
metrics_capped = [col+'_capped' for col in metrics_to_cap]

In [12]:
df[metrics_capped] = cap_above_quantile(df, metrics=metrics_to_cap, q=0.999)

In [13]:
print("Initial mean = \t\t", df[metrics_to_cap].mean().values[0])
print("Mean after capping = \t", df[metrics_capped].mean().values[0])

Initial mean = 		 2.1717967967967966
Mean after capping = 	 2.1697102102102104


<br><br>Function help:

In [14]:
remove_outliers?

Signature: remove_outliers(df, metrics, contamination=0.005)
Docstring:
Removes outliers from a DataFrame using the Isolation Forest algorithm.

Parameters
----------
df : pandas.DataFrame
    Input DataFrame containing the data to be filtered.
metrics : list of str
    List of column names used as features for outlier detection.
contamination : float, optional
    The proportion of observations in the data expected to be outliers.
    Default is 0.005 (i.e., 0.5%).

Returns
-------
pandas.DataFrame
    A filtered DataFrame with outlier rows removed, retaining the original columns.

Notes
-----
This function applies the Isolation Forest algorithm to detect anomalies
based on the selected metric columns. Outliers are identified as data points
for which the model prediction equals -1, and are excluded from the returned result.
File:      ~/.conda/envs/oyaksin/lib/python3.11/site-packages/abiasales/cleaning.py
Type:      function

<br><br>Function help with the code:

In [15]:
remove_outliers??

Signature: remove_outliers(df, metrics, contamination=0.005)
Source:   
def remove_outliers(df, metrics, contamination=0.005):
    """
    Removes outliers from a DataFrame using the Isolation Forest algorithm.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing the data to be filtered.
    metrics : list of str
        List of column names used as features for outlier detection.
    contamination : float, optional
        The proportion of observations in the data expected to be outliers.
        Default is 0.005 (i.e., 0.5%).

    Returns
    -------
    pandas.DataFrame
        A filtered DataFrame with outlier rows removed, retaining the original columns.

    Notes
    -----
    This function applies the Isolation Forest algorithm to detect anomalies
    based on the selected metric columns. Outliers are identified as data points
    for which the model prediction equals -1, and are excluded from the returned result.
    """

    df_new = df

# WEIGHTS

Calculation of Weights for Ratio Metrics
There are four methods in total:
- **uniform** – all weights are equal to 1. Essentially, this corresponds to computing the metric at the user level and then taking the average of per-user metrics.
- **size** – all weights are equal to the denominator values. This is equivalent to summing the numerator across all users and dividing by the sum of denominators across all users.
- **sqrt** – weights are equal to the square root of the denominator.
- **intra_corr** – intra-user correlation–aware weighting, which accounts for within-user correlation. See https://arxiv.org/abs/1911.03553.

In [16]:
calc_weights(df['clicks'], df['views'], weight_method='uniform')

array([1., 1., 1., ..., 1., 1., 1.], shape=(1998000,))

In [17]:
calc_weights(df['clicks'], df['views'], weight_method='size')

array([ 6., 10., 13., ..., 71.,  1.,  1.], shape=(1998000,))

In [18]:
calc_weights(df['clicks'], df['views'], weight_method='sqrt')

array([2.44948974, 3.16227766, 3.60555128, ..., 8.42614977, 1.        ,
       1.        ], shape=(1998000,))

In [19]:
calc_weights(df['clicks'], df['views'], weight_method='intra_corr')

array([ 5.80301791,  9.4241773 , 12.02070401, ..., 48.12819504,
        1.        ,  1.        ], shape=(1998000,))

# STATS

Functions for calculating the mean, standard deviation, confidence interval using the delta method, and the linearization coefficient.

In [20]:
df['ratios'] = df['clicks'] / df['views']
df['weight'] = calc_weights(df['clicks'], df['views'], weight_method='intra_corr')

Weighted mean

In [21]:
weighted_mean(df['ratios'], df['weight'])

np.float64(0.13781618149585192)

Weighted standard deviation

In [22]:
weighted_std(df['ratios'], df['weight'])

np.float64(0.10983297618611822)

General function for calculating the standard deviation

In [23]:
# Without weights
calc_std(num=df['clicks'], den=df['views'], weights=None, use_delta_method=False)

np.float64(0.23214760147766075)

In [24]:
# With weights
calc_std(num=df['clicks'], den=df['views'], weights=df['weight'], use_delta_method=False)

np.float64(0.10983297618611822)

In [25]:
# Using Delta-method
calc_std(num=df['clicks'], den=df['views'], use_delta_method=True)

np.float64(0.1134948680731779)

Compute the confidence interval using the delta method. Required when estimating the confidence interval of a relative effect in an experiment.

In [26]:
calc_delta_method_ratio_confint(
    mean_1=0.50, std_1=2, nobs_1=100000,
    mean_2=0.55, std_2=2, nobs_2=120000,
    alpha=0.05
)

(np.float64(0.06464130734476425), np.float64(0.13551869265523592))

Linearization coefficient for transforming ratio metric into linear

In [27]:
calc_linearization_coef(num=df['clicks'], den=df['views'], weights=df['weight'])

np.float64(0.13933058456142375)

# STRATIFICATION

Adding strata column (always named 'unified_strata')

In [28]:
add_strata_col(df=df, strats_cols=STRATS_COLS)

In [29]:
df['unified_strata'].head()

0     (desktop, RU)
1      (mobile, RU)
2    (desktop, INT)
3     (desktop, RU)
4     (mobile, INT)
Name: unified_strata, dtype: object

Calculating stratas weights

In [30]:
calc_strats_weights(df=df, strats_cols=STRATS_COLS)

{('desktop', 'RU'): np.float64(0.2499854854854855),
 ('mobile', 'RU'): np.float64(0.2500675675675676),
 ('desktop', 'INT'): np.float64(0.24958908908908908),
 ('mobile', 'INT'): np.float64(0.25035785785785786)}

# METRICS

General functions for calculating the mean and standard deviation with stratification, CUPED, or linearization adjustments.

Calculation of the mean and standard deviation accounting for stratification and the delta method, using precomputed weights.

In [31]:
calc_mean_and_std(
    df=df,
    metric_num='clicks',
    metric_den='views',
    metric_weight='weight',
    use_delta_method=False,
    use_stratification=False,
    strats_cols=None,
    strats_weights=None
)

(np.float64(0.13781618149585192), np.float64(0.10983297618611822))

A complete pipeline for calculating the mean and standard deviation, accounting for:
- weight computation
- stratification
- Delta method
- inearization
- CUPED

In [32]:
calc_pipeline_mean_and_std(
    df=df,
    metric_num='clicks',
    metric_den='views',
    weight_method='intra_corr',
    apply_linearization=False,
    use_delta_method=False,
    use_stratification=False,
    strats_cols=None,
    var_reduction_method=None,
    var_reduction_covariates=None
)

(np.float64(0.13781618149585192), np.float64(0.10983297618611822))

In [33]:
# Another weighting and using Delta method
calc_pipeline_mean_and_std(
    df=df,
    metric_num='clicks',
    metric_den='views',
    weight_method='uniform',
    apply_linearization=False,
    use_delta_method=True,
    use_stratification=False,
    strats_cols=None,
    var_reduction_method=None,
    var_reduction_covariates=None
)

(np.float64(0.13657568137405623), np.float64(0.1134948680731779))

A complete pipeline for calculating the mean and standard deviation, accounting for:
- weight computation
- stratification
- Delta method
- inearization
- CUPED

Returns not only mean and std, but also weights with calculated ratios.

In [34]:
mean_a, std_a, weights_a, ratios_a, mean_b, std_b, weights_b, ratios_b = calc_groups_stats(
    df_control=df[df['exp_group']=='A'],
    df_treatment=df[df['exp_group']=='B'],
    metric_num='clicks',
    metric_den='views',
    weight_method='uniform',
    apply_linearization=False,
    use_delta_method=True,
    use_stratification=False,
    strats_cols=None,
    var_reduction_method=None,
    var_reduction_covariates=None
)

In [35]:
mean_a, std_a, mean_b, std_b

(np.float64(0.1303139341391071),
 np.float64(0.10980255229376944),
 np.float64(0.1428419055717696),
 np.float64(0.11553531566154163))

# INFERENCE

Check that the proportion of the control group does not deviate significantly from the expected level.

In [36]:
check_groups_distribution(df=df, exp_group_col='exp_group', control_name='A', control_perc=0.5)

,
Expected control group %,0.50
Fact control group %,0.50
Difference p-value,0.61


Statistical test for proportions, returning the p-value and the confidence interval of the uplift.

In [37]:
calc_stattest_proportion_test(
    proportion_1=df.loc[df['exp_group']=='A', 'payer_flag'],
    proportion_2=df.loc[df['exp_group']=='B', 'payer_flag'],
    uplift_type='rel', # 'rel' – относительный аплифт, 'abs' – абсолютный аплифт
    confint_alpha=0.05 # значимость доверительного интервала
)

(np.float64(0.12539451489114234),
 (np.float64(-0.00852837133653972), np.float64(0.0010488204865799489)))

z-test, returning the p-value and the confidence interval of the uplift.

In [38]:
mean_a, std_a, _, _, mean_b, std_b, _, _ = calc_groups_stats(
    df_control=df[df['exp_group']=='A'],
    df_treatment=df[df['exp_group']=='B'],
    metric_num='payer_flag'
)

In [39]:
calc_stattest_z_test(
    mean_1=mean_a, # mean of the first group
    std_1=std_a, # standard deviation of the first group
    nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']), # number of observations in the first group
    mean_2=mean_b, # mean of the second group
    std_2=std_b, # standard deviation of the second group
    nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']), # number of observations in the second group
    uplift_type='rel', # 'rel' – relative uplift, 'abs' –absolute uplift
    confint_alpha=0.05 # significance level of confidence interval
)

(np.float64(0.12539373961554157),
 (np.float64(-0.008538381686063651), np.float64(0.0010387851703516611)))

t-test, returning the p-value and the confidence interval of the uplift.

In [40]:
calc_stattest_t_test(
    mean_1=mean_a, # mean of the first group
    std_1=std_a, # standard deviation of the first group
    nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']), # number of observations in the first group
    mean_2=mean_b, # mean of the second group
    std_2=std_b, # standard deviation of the second group
    nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']), # number of observations in the second group
    uplift_type='rel', # 'rel' – relative uplift, 'abs' –absolute uplift
    confint_alpha=0.05 # significance level of confidence interval
)

(np.float64(0.1253940672322704),
 (np.float64(-0.008538381686063651), np.float64(0.0010387851703516611)))

bootstrap, returning the p-value and the confidence interval of the uplift.

In [41]:
calc_stattest_bootstrap(
    metric_1=df[df['exp_group']=='A'].sample(100000).reset_index()['payer_flag'],
    weights_1=None,
    metric_2=df[df['exp_group']=='B'].sample(100000).reset_index()['payer_flag'],
    weights_2=None,
    uplift_type='rel', # 'rel' – relative uplift, 'abs' –absolute uplift
    confint_alpha=0.05, # significance level of confidence interval
    n_bootstrap=500 # number of bootstrap iterations
)

(np.float64(0.572),
 (np.float64(-0.010673550471367798), np.float64(0.019530885027885754)))

General wrapper for all statistical tests.

In [42]:
calc_stattest(
    metric_1=df.loc[df['exp_group']=='A', 'payer_flag'],
    metric_2=df.loc[df['exp_group']=='B', 'payer_flag'],
    stat_method='proportion', uplift_type='rel', confint_alpha=0.05, n_bootstrap=500
)

,mean_control,mean_treatment,uplift,confint,pvalue
0,0.25,0.25,-0.38%,"(-0.85%, 0.1%)",0.13


In [43]:
calc_stattest(
    mean_1=mean_a, std_1=std_a, nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']),
    mean_2=mean_b, std_2=std_b, nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']),
    stat_method='t_test', uplift_type='rel', confint_alpha=0.05
)

,mean_control,mean_treatment,uplift,confint,pvalue
0,0.25,0.25,-0.38%,"(-0.85%, 0.1%)",0.13


In [44]:
calc_stattest(
    mean_1=mean_a, std_1=std_a, nobs_1=len(df.loc[df['exp_group']=='A', 'payer_flag']),
    mean_2=mean_b, std_2=std_b, nobs_2=len(df.loc[df['exp_group']=='B', 'payer_flag']),
    stat_method='z_test', uplift_type='rel', confint_alpha=0.05
)

,mean_control,mean_treatment,uplift,confint,pvalue
0,0.25,0.25,-0.38%,"(-0.85%, 0.1%)",0.13


In [45]:
calc_stattest(
    metric_1=df[df['exp_group']=='A'].sample(100000).reset_index()['payer_flag'],
    metric_2=df[df['exp_group']=='B'].sample(100000).reset_index()['payer_flag'],
    stat_method='bootstrap', uplift_type='rel', confint_alpha=0.05, n_bootstrap=500
)

,mean_control,mean_treatment,uplift,confint,pvalue
0,0.25,0.25,-0.15%,"(-1.51%, 1.34%)",0.84


# POWER

Sample size calculation accounting for stratification and CUPED.

In [46]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,30367


In [47]:
# WITH STRATIFICATION
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,26731


In [48]:
# WITH CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,27662


In [49]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='sample_size',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], uplift=[0.05], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
5.0%,23058


MDE calculation accounting for sample size

In [50]:
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[30345, 23009], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
23009,5.7%
30345,5.0%


In [51]:
# WITH STRATIFICATION AND CUPED
calc_power_table(
    df=df,
    metric_num='purchases',
    mode='mde',
    use_stratification=True,
    strats_cols=STRATS_COLS,
    var_reduction_method='cuped',
    var_reduction_covariates=FEATURE_COLS,
    alpha=[0.05], power=[0.8], sample_size=[30345, 23009], control_perc=0.5
)

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
23009,5.0%
30345,4.4%


# RESULTS

In [6]:
df_a = create_test_data(uplift=0, n_users=30000)
df_a['exp_group'] = 'A'
df_b = create_test_data(uplift=0.05, n_users=30000)
df_b['exp_group'] = 'B'

df_results = pd.concat([df_a, df_b]).reset_index(drop=True)
del df_a, df_b

List of metric config parameters:
- **num** - name of the column containing the numerator of the metric.
- **den** - name of the column containing the denominator of the metric. Defaults to None and can be omitted.
- **weight_method** - type of user weighting. Default is 'uniform'. Possible values: 'uniform', 'size', 'sqrt', 'intra_corr'.
- **stat_method** - statistical test used to calculate the p-value. Default is 't_test'. Possible values: 'z_test', 't_test', 'proportion' (for conversion metrics only), 'bootstrap'.
- **use_delta_method** - whether to use the delta method for variance estimation (relevant only for ratio metrics).
- **var_reduction_method** - variance reduction method. Default is None. Possible values: None, 'cuped', 'cupac'.
- **var_reduction_covariates** - list of covariates for CUPED/CUPAC. Defaults to None. Ignored if var_reduction_method = None.
- **handle_outliers** - how to handle outliers. Default is None. Posible values: remove (IsolationTree), cap (capping values).
- **outliers_contamination** - threshold to define an outlier. Default is 0.001. In case of handle_outliers="remove" – top outliers_contamination % of samples, in case of handle_outliers="cap" (capping values) – top quantile level of capping.
- **outliers_cols** – columns to detect outliers, relevant only for handle_outliers="remove".

Stratification is configured at the calc_exp_results function level:
- **use_stratification** - whether to apply stratification. Defaults to False.
- **strats_cols** - list of columns used for stratification. Defaults to None; ignored if use_stratification = False.

The function returns two p-value values:
- **pvalue_init** – the initial p-value obtained from the statistical test.
- **pvalue** – the p-value adjusted for multiple hypothesis testing.

In [7]:
# Metrics
metrics = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test'
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test',
        'use_delta_method': True
    }
}

# Same metrics but with CUPED and outliers handling
metrics_cuped = {
    'purchases': {
        'num': 'purchases',
        'weight_method': 'uniform',
        'stat_method': 't_test',
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'remove',
        'outliers_cols': FEATURE_COLS,
        'outliers_contamination': 0.005
    },
    'CTR': {
        'num': 'clicks',
        'den': 'views',
        'weight_method': 'size',
        'stat_method': 't_test',
        'use_delta_method': True,
        'var_reduction_method': 'cuped',
        'var_reduction_covariates': FEATURE_COLS,
        'handle_outliers': 'cap',
        'outliers_contamination': 0.01
    }
}

In [8]:
results = calc_exp_results(
    df_results, exp_group_col='exp_group', metrics=metrics
)

,metric,group_1,group_2,mean_control,mean_treatment,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.066200,2.140167,3.58%,"('-0.1%', '7.28%')",0.052983,0.052983
1,CTR,A,B,0.131584,0.137934,4.83%,"('3.45%', '6.2%')",0.000000,0.000000


In [9]:
# WITH STRATIFICATION
results = calc_exp_results(
    df, exp_group_col='exp_group', metrics=metrics,
    use_stratification=True, strats_cols=STRATS_COLS
)

,metric,group_1,group_2,mean_control,mean_treatment,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.081367,2.276153,9.36%,"('8.73%', '9.98%')",0.000000,0.000000
1,CTR,A,B,0.130095,0.142786,9.75%,"('9.53%', '9.98%')",0.000000,0.000000


In [10]:
# WITH CUPED
results = calc_exp_results(
    df, exp_group_col='exp_group', metrics=metrics_cuped
)

,metric,group_1,group_2,mean_control,mean_treatment,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.068860,2.196002,6.15%,"('5.52%', '6.77%')",0.000000,0.000000
1,CTR,A,B,0.130495,0.138331,6.0%,"('5.7%', '6.31%')",0.000000,0.000000


In [11]:
# WITH STRATIFICATION AND CUPED
results = calc_exp_results(
    df, exp_group_col='exp_group', metrics=metrics_cuped,
    use_stratification=True, strats_cols=STRATS_COLS
)

,metric,group_1,group_2,mean_control,mean_treatment,uplift,confint,pvalue_init,pvalue
0,purchases,A,B,2.068115,2.202331,6.49%,"('5.91%', '7.07%')",0.000000,0.000000
1,CTR,A,B,0.128857,0.136869,6.22%,"('5.92%', '6.52%')",0.000000,0.000000


# SIMULATION

Simulate:
- A/A tests to verify that the p-value properly controls the Type I error rate.
- A/B tests to compare the statistical power of different tests.

Each of the functions **`run_aa_test_simulation`** and **`run_ab_test_simulation`** takes a **`configs`** argument — a list of dictionaries, where each dictionary defines the metric configuration and the statistical methods to be used.  

Each dictionary has the following **structure**:

---

- **`metric_num`** *(str)*  
  Name of the column with the metric’s numerator (e.g., `"purchases"`).

- **`metric_den`** *(str)*  
  Name of the column with the metric’s denominator.  
  If `None`, assumes a value of 1.

- **`weight_method`** *(str)*  
  Unit-level weighting scheme to apply.  
  Allowed:  
  - `'uniform'`: all users weight = 1  
  - `'size'`: weights equal to the metric “size” (e.g., denominator or exposure)  
  - `'sqrt'`: weights are √(size)  
  - `'intra_corr'`: correlation-aware weights (accounts for within-user correlation)  
  Default: `'uniform'`.

- **`stat_method`** *(str)*  
  Statistical test to compute the p-value.  
  Allowed: `'proportion'`, `'z_test'`, `'t_test'`, `'bootstrap'`.  
  Use `'proportion'` only for conversion metrics.  
  Default: `'t_test'`.

- **`use_stratification`** *(bool)*  
  Whether to apply stratification.  
  If `False`, `strats_cols` are ignored.  
  Default: `False`.

- **`strats_cols`** *(list[str] | None)*  
  Columns to use as strata keys (e.g., `device`, `country`).  
  Ignored if `use_stratification=False`.  
  Default: `None`.

- **`var_reduction_method`** *(str)*  
  Variance reduction method.  
  Allowed: `None`, `'cuped'`, `'cupac'`.  
  Default: `None`.

- **`var_reduction_covariates`** *(list[str] | None)*  
  Covariates for CUPED/CUPAC.  
  Ignored if `var_reduction_method=None`.  
  Default: `None`.

- **`handle_outliers`** *(str | None)*  
  Strategy for outliers.  
  Allowed:  
  - `None`  
  - `'remove'`: drop outliers using IsolationTree  
  - `'cap'`: cap metric at quantile threshold  
  Default: `None`.

- **`outliers_contamination`** *(float | None)*  
  Expected outlier fraction (e.g., `0.001` = 0.1%).  
  Used to set cutoffs for `'remove'`/`'cap'`.  
  Ignored if `handle_outliers=None`.  
  Default: `0.01`.

- **`outliers_cols`** *(list[str])*  
  Columns on which to detect/handle outliers (e.g., `exposure`, `spend`, `price`).  
  Ignored if `handle_outliers != 'remove'`.

---

**Example:**

```python
{
    'metric_num': 'purchases',
    'weight_method': 'uniform',
    'stat_method': 'z_test',
    'use_stratification': False,
    'strats_cols': STRATS_COLS,                       # ignored because use_stratification=False
    'var_reduction_method': ['country'],
    'var_reduction_covariates': ['purchases_cov'],    # ignored because var_reduction_method=None
    'handle_outliers': None,
    'outliers_contamination': 0.001,                  # ignored because handle_outliers=None
    'outliers_cols': ['purchases_cov']                # ignored because handle_outliers=None
}


In [6]:
df_sim = df[df['exp_group']=='A']
n_exp = 1000
sample_size = 1000

Estimate the MDE (Minimum Detectable Effect) for a given sample size.

In [7]:
pt, pw = calc_power_table(
    df=df_sim,
    metric_num='purchases',
    mode='mde',
    alpha=[0.05], power=[0.8], sample_size=[sample_size], control_perc=0.5,
    return_power_list=True
)
display(pt)

model_uplift = pw[0]

"Errors ($\alpha$, $\beta$)",(0.05; 0.2)
1000,27.8%


Construct a set of statistical tests to be compared.

In [8]:
configs = construct_exp_configs(
    metric_num='purchases', # Numerator of metric
    metric_den=None, # Denominator of metric
    weight_methods=['uniform'], # 'uniform', 'size', 'sqrt', 'intra_corr'
    stat_methods=['t_test'], # 'z_test', 't_test', 'bootstrap'
    use_stratification=[False, True], # False, True
    strats_cols=STRATS_COLS,
    var_reduction_methods=[None, 'cuped'], # None, 'cuped', 'cupac'
    var_reduction_covariates=FEATURE_COLS,
    handle_outliers=[None], # None, 'remove', 'cap',
    outliers_contamination=[0.001], # Default is 0.01
    outliers_cols=FEATURE_COLS
)

In [11]:
configs

[{'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': None,
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'handle_outliers': None,
  'outliers_contamination': 0.001,
  'outliers_cols': []},
 {'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test',
  'use_stratification': False,
  'strats_cols': ['platform', 'country'],
  'var_reduction_method': 'cuped',
  'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'],
  'handle_outliers': None,
  'outliers_contamination': 0.001,
  'outliers_cols': []},
 {'num': 'purchases',
  'den': None,
  'weight_method': 'uniform',
  'apply_linearization': False,
  'use_delta_method': False,
  'stat_method': 't_test'

Generate synthetic A/A tests

In [9]:
results_aa = run_aa_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp
)

Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:14<00:00, 68.75it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:47<00:00, 20.95it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:19<00:00, 51.15it/s]


Running A/A for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:52<00:00, 18.90it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,fpr-pvalue_auc,pvalues_aa_uniform
0,uniform,False,False,t_test,False,None,None,0.001000,0.496447,True
1,uniform,False,False,t_test,False,cuped,None,0.001000,0.497743,True
2,uniform,False,False,t_test,True,None,None,0.001000,0.503941,True
3,uniform,False,False,t_test,True,cuped,None,0.001000,0.496726,True


All statistical tests are valid

Generate synthetic A/B tests

In [10]:
results_ab = run_ab_test_simulation(
    df_sim,
    configs,
    control_sample_size=sample_size,
    n_exp=n_exp,
    metric_type='value',   # Metric type – 'proportion', 'value', 'ratio',
                           # Uplifts will be generated differently depending on it.
    uplift=model_uplift
)

Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:19<00:00, 52.57it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': False, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:52<00:00, 18.99it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': None, 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:24<00:00, 41.39it/s]


Running A/B for config: {'num': 'purchases', 'den': None, 'weight_method': 'uniform', 'apply_linearization': False, 'use_delta_method': False, 'stat_method': 't_test', 'use_stratification': True, 'strats_cols': ['platform', 'country'], 'var_reduction_method': 'cuped', 'var_reduction_covariates': ['views_cov', 'clicks_cov', 'purchases_cov'], 'handle_outliers': None, 'outliers_contamination': 0.001, 'outliers_cols': [], 'uplift_type': 'rel'}...


100%|██████████| 1000/1000 [00:57<00:00, 17.29it/s]


,weight,apply_lin,use_delta_method,stat,use_strat,use_var_reduction,handle_outliers,outliers_contamination,power
0,uniform,False,False,t_test,False,None,None,0.001000,0.787000
1,uniform,False,False,t_test,False,cuped,None,0.001000,0.834000
2,uniform,False,False,t_test,True,None,None,0.001000,0.819000
3,uniform,False,False,t_test,True,cuped,None,0.001000,0.889000
